Fraudulent_Transaction_Detection_For_Finlora_Company 

In [1]:
# Importing the python libraries
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV 


In [2]:
# Loading the dataset

customer_transaction_data = pd.read_csv(r"C:/Users/USER/Desktop/AMDARI/Fraudulent_Transaction_Detection_For_Finlora_Company/Fraudulent_Transaction_Detection_For_Finlora_Company/Finlora Dataset/FinLora_Customer_Transaction_Dataset.csv")
customer_transaction_data.head() 

,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,...,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud
0,fee8542d-8ee6-4b0d-9671-c294dd08ed26,402cccc9-28de-45b3-9af7-cc5302aa1f93,2022-10-03 18:40:59.468549+00:00,US,USD,CAD,ATM,278.19,278.19,4.25,...,0.123,standard,263,0.522,0,0.223,0,0,0.0,0
1,bfdb9fc1-27fe-4a85-b043-4d813d679259,67c2c6b3-ef0a-4777-a3f1-c84a851bb6ad,2022-10-03 20:39:38.468549+00:00,CA,CAD,MXN,web,208.51,154.29,4.24,...,0.569,standard,947,0.475,0,0.268,0,1,0.0,0
2,fc855034-3ea5-4993-9afa-b511d93fe5e8,6d0d9b27-fa26-45f8-93b1-2df29d182d9c,2022-10-03 23:02:43.468549+00:00,US,USD,CNY,mobile,160.33,160.33,2.70,...,0.437,enhanced,367,0.939,0,0.176,0,0,0.0,0
3,2cf8c08e-42ec-444d-a755-34b9a2a0a4ca,7bd5200c-5d19-44f0-9afe-8b339a05366b,2022-10-04 01:08:53.468549+00:00,US,USD,EUR,mobile,59.41,59.41,2.22,...,0.594,standard,147,0.551,0,0.391,0,0,0.0,0
4,d907a74d-b426-438d-97eb-dbe911aca91c,70a93d26-8e3a-4179-900c-a4a7a74d08e5,2022-10-04 09:35:03.468549+00:00,US,USD,INR,mobile,200.96,200.96,3.61,...,0.121,enhanced,257,0.894,0,0.257,0,0,0.0,0


In [3]:
# Checking for number of rows and columns

customer_transaction_data.shape 

(11400, 26)

In [4]:
# Checking for all the columns in the dataset

customer_transaction_data.columns 

Index(['transaction_id', 'customer_id', 'timestamp', 'home_country',
       'source_currency', 'dest_currency', 'channel', 'amount_src',
       'amount_usd', 'fee', 'exchange_rate_src_to_dest', 'device_id',
       'new_device', 'ip_address', 'ip_country', 'location_mismatch',
       'ip_risk_score', 'kyc_tier', 'account_age_days', 'device_trust_score',
       'chargeback_history_count', 'risk_score_internal', 'txn_velocity_1h',
       'txn_velocity_24h', 'corridor_risk', 'is_fraud'],
      dtype='object')

In [ ]:
# Checking the missing data

customer_transaction_data.isnull().sum() 

transaction_id                 0
customer_id                    0
timestamp                     29
home_country                   0
source_currency                0
dest_currency                  0
channel                        0
amount_src                     0
amount_usd                   305
fee                          295
exchange_rate_src_to_dest      0
device_id                      0
new_device                     0
ip_address                   305
ip_country                   301
location_mismatch              0
ip_risk_score                  0
kyc_tier                     300
account_age_days               0
device_trust_score           295
chargeback_history_count       0
risk_score_internal            0
txn_velocity_1h                0
txn_velocity_24h               0
corridor_risk                  0
is_fraud                       0
dtype: int64

Checking the missing values from above variable, we can confirm that:

. timestamp has 29 missing values
. amount_usd has 305 missing values
. fee has 295 missing values
. ip_address has 305 missing values
. ip_country has 301 missing values
. kyc_tier has 300 missing values
. device_trust_score has 295 missing values


In [17]:
# Create the initial data_dict DataFrame

import pandas as pd

data_dict = pd.DataFrame({
    'Column Name': customer_transaction_data.columns,
    'Data Type': customer_transaction_data.dtypes.astype(str),
    'Sample Value': customer_transaction_data.iloc[0].values,
})

# Define custom column descriptions

descriptions = { 
    'transaction_id': 'Unique transaction identifier',
    'customer_id': 'Unique customer identifier',
    'timestamp': 'Date and time transaction',
    'home_country': 'Country of customer residence',
    'source_currency': 'Original currency usd',
    'dest_currrency': 'Currency received by destination',
    'channel': 'Platform used (mobile, web, API)',
    'amount_usd': 'Transaction amount in USD',
    'is_fraud': 'Fraud flag (1 = Yes, 0 =No)' 

}

# Map description and display 

data_dict['Description'] = data_dict['Column Name'].map(descriptions).fillna('N/A')
data_dict = data_dict[['Column Name', 'Data Type', 'Description', 'Sample Value']]

display(data_dict) 

,Column Name,Data Type,Description,Sample Value
transaction_id,transaction_id,object,Unique transaction identifier,fee8542d-8ee6-4b0d-9671-c294dd08ed26
customer_id,customer_id,object,Unique customer identifier,402cccc9-28de-45b3-9af7-cc5302aa1f93
timestamp,timestamp,"datetime64[ns, UTC]",Date and time transaction,2022-10-03 18:40:59.468549+00:00
home_country,home_country,object,Country of customer residence,US
source_currency,source_currency,object,Original currency usd,USD
dest_currency,dest_currency,object,N/A,CAD
channel,channel,object,"Platform used (mobile, web, API)",ATM
amount_src,amount_src,float64,N/A,278.19
amount_usd,amount_usd,float64,Transaction amount in USD,278.19
fee,fee,float64,N/A,4.25


In [6]:
# Checking the data info

customer_transaction_data.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11400 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   transaction_id             11400 non-null  object 
 1   customer_id                11400 non-null  object 
 2   timestamp                  11371 non-null  object 
 3   home_country               11400 non-null  object 
 4   source_currency            11400 non-null  object 
 5   dest_currency              11400 non-null  object 
 6   channel                    11400 non-null  object 
 7   amount_src                 11400 non-null  object 
 8   amount_usd                 11095 non-null  float64
 9   fee                        11105 non-null  float64
 10  exchange_rate_src_to_dest  11400 non-null  float64
 11  device_id                  11400 non-null  object 
 12  new_device                 11400 non-null  bool   
 13  ip_address                 11095 non-null  obj

KEY OBSERVATION OF THE DATASET

A. The dataset contains 11,400 rows and 26 columns representing customer finacial transactions

B. Missing data is concentrated across 7 specific columns:
   - amount_usd and ip_address: 305 missing values each
   - ip_country: 301 missing values
   - kyc_tier: 300 missing values
   - fee and device_trust_score: 295 missing values each
   - timestamp: 29 missing values

C. The features distribution consist of mixed data types:
   - 12 Object/String variables(e.g., transaction_id, customer_id, home_country, channel)
   - 7 Float64 numerical variables(e.g., amount_usd, fee, exchange_rate_src_to_dest)
   - 5 Int64 numerical variables(e.g., account_age_days, is_fraud)
   - 2 Boolean binary flags(new_device, location_mismatch)

D. Complete Fields:
   - Primary identifiers and structural columns like transaction_id, customer_id, channel, home_country, and the target variable is_fraud have 0 missing values

E. Data Type Convesion Requireds:
   - timestamp was initially loaded as an object (string) and needs to be converted to datetime64.
   - amount_src required conversion to numeric (float) using pd.to_numeric() due to string/mixed formatting.


In [10]:
# Checking for Label imbalance

customer_transaction_data['is_fraud'].value_counts(normalize=True).reset_index() 

,is_fraud,proportion
0,0,0.912544
1,1,0.087456


In [11]:
# Converting data types of timestamp to datetime

customer_transaction_data['timestamp'] = pd.to_datetime(customer_transaction_data['timestamp'],errors='coerce')

# Converting data type of income_usd to float 

customer_transaction_data['amount_src'] = pd.to_numeric(customer_transaction_data['amount_src'],errors='coerce')

In [12]:
# Checking the dataset again to confirm the data type

customer_transaction_data.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11400 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype              
---  ------                     --------------  -----              
 0   transaction_id             11400 non-null  object             
 1   customer_id                11400 non-null  object             
 2   timestamp                  11339 non-null  datetime64[ns, UTC]
 3   home_country               11400 non-null  object             
 4   source_currency            11400 non-null  object             
 5   dest_currency              11400 non-null  object             
 6   channel                    11400 non-null  object             
 7   amount_src                 11396 non-null  float64            
 8   amount_usd                 11095 non-null  float64            
 9   fee                        11105 non-null  float64            
 10  exchange_rate_src_to_dest  11400 non-null  float64            
 11  de

A. Data Type Conversion: timestamp was successfully converted to datetime64[ns, UTC], and amount_src was coerce to float64

B. Coercion Side-Effect (New Nulls Introduced): 
   - Converting timestamp with error='coerce' generated 61 nulls(up from 29), indicating 32 invalid date format strings were converted to NaT.
   - Converting amount_src to float created 4 missing values from non-numeric strings.

In [14]:
# Rechecking for missing data

customer_transaction_data.isnull().sum()  

transaction_id                 0
customer_id                    0
timestamp                     61
home_country                   0
source_currency                0
dest_currency                  0
channel                        0
amount_src                     4
amount_usd                   305
fee                          295
exchange_rate_src_to_dest      0
device_id                      0
new_device                     0
ip_address                   305
ip_country                   301
location_mismatch              0
ip_risk_score                  0
kyc_tier                     300
account_age_days               0
device_trust_score           295
chargeback_history_count       0
risk_score_internal            0
txn_velocity_1h                0
txn_velocity_24h               0
corridor_risk                  0
is_fraud                       0
dtype: int64

Missing Data Imputation Strategy

A. Smart Feature Imputation
   - amount_usd: Fill missing values (305) by multiplying `amount_src` by `exchange_rate_src_to_dest`.
   - fee: Fill missing values (295) using the median `fee` grouped by `channel`. Fill any remaing with the overall median.
   - ip_country: Fill missing values (301) using the value from `home_country`.
   - kyc_tier: Fill missing values (300) with the mode (most frequent value) of `kyc_tier`.
   - device_trust_score: Fill missing values (295) using the median grouped by `kyc_tier` and 'new_device', falling back to the overall median.

B. Row Elimination (dropna)
   - ip_address: Drop rows with missing values (305) because IP address are unique identifiers and cannot be accurately imputed.
   - timestamp & amount_src: Drop rows with missing values (61 and 4 respective) using `dropna()`, as these represent a negligible percentage of the total dataset.


In [16]:
# Calculating exchange rates per curency

exchange_rates = customer_transaction_data[customer_transaction_data['amount_usd'].notna()].groupby('source_currency').apply(
    lambda x: (x['amount_usd'] / x['amount_src']).mean()
).to_dict()
exchange_rates  

C:\Users\USER\AppData\Local\Temp\ipykernel_2688\165734851.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  exchange_rates = customer_transaction_data[customer_transaction_data['amount_usd'].notna()].groupby('source_currency').apply(


{'CAD': 0.7216095926871465,
 'GBP': 1.223441221648679,
 'USD': 0.9838730321259439}

In [31]:
# filling the missing values of the amount_usd
 
customer_transaction_data['amount_usd']=customer_transaction_data.apply( 
    lambda row: row['amount_usd'] if pd.notna(row['amount_usd']) else row['amount_src'] * exchange_rates.get(row['source_currency'],1),
    axis=1
)

In [32]:
# filling the missing values of the fee variable

if 'fee' in customer_transaction_data.columns:
    if 'channel' in customer_transaction_data.columns:
        customer_transaction_data['fee'] = customer_transaction_data.groupby('channel')['fee'].transform(lambda y: y.fillna(y.median()))
        customer_transaction_data['fee'] = customer_transaction_data['fee'].fillna(customer_transaction_data['fee'].median()) 

In [33]:
# filling the missing values of ip_country variable

if {'ip_country)', 'home_country'}.issubset(customer_transaction_data.columns):
    customer_transaction_data['ip_country'] = customer_transaction_data['ip_country'].fillna(customer_transaction_data['home_contry']) 

In [34]:
# filling the missing values of device_trust_score variable
 
if 'device_trust_score' in customer_transaction_data.columns:
    if {'new_device', 'kyc_tier'}.issubset(customer_transaction_data.columns):
        customer_transaction_data['device_trust_score']=customer_transaction_data.groupby(['new_device', 'kyc_tier'])['device_trust_score'].transform(lambda x: x.fillna(x.median()))
        customer_transaction_data['device_trust_score'] =customer_transaction_data['device_trust_score'].fillna(customer_transaction_data['device_trust_score'].median()) 

In [35]:
# droping the timestamp, amount_src and ip_address missing values

customer_transaction_data.dropna(subset=['timestamp', 'amount_src', 'ip_address'], inplace=True) 

In [36]:
# rechecking for missing values

customer_transaction_data.isnull().sum()  

transaction_id               0
customer_id                  0
timestamp                    0
home_country                 0
source_currency              0
dest_currency                0
channel                      0
amount_src                   0
amount_usd                   0
fee                          0
exchange_rate_src_to_dest    0
device_id                    0
new_device                   0
ip_address                   0
ip_country                   0
location_mismatch            0
ip_risk_score                0
kyc_tier                     0
account_age_days             0
device_trust_score           0
chargeback_history_count     0
risk_score_internal          0
txn_velocity_1h              0
txn_velocity_24h             0
corridor_risk                0
is_fraud                     0
dtype: int64

# STEP TO TAKE TO DO THE SANITY CHECK
* Check for invalid number i.e negative numbers for variable that contains age, monetary values, velocity etc
* Check if transaction timestamp appear in the future
* Check that values falls between range
* Inspect for inconsistency among variable values in the dataset 

In [39]:
# Checking negative values in numeric columns

negative_counts = {
    'amount_src': (customer_transaction_data['amount_src'] < 0).sum(),
    'amount_usd': (customer_transaction_data['amount_usd'] < 0).sum(),
    'fee': (customer_transaction_data['fee'] < 0).sum(),
    'device_trust_score': (customer_transaction_data['device_trust_score'] < 0).sum(),
    'txn_velocity_1h': (customer_transaction_data['txn_velocity_1h'] < 0).sum(),
    'txn_velocity_24h': (customer_transaction_data['txn_velocity_24h'] < 0).sum(),
    'risk_score_internal': (customer_transaction_data['risk_score_internal'] < 0).sum(),
 
}
negative_counts  

{'amount_src': np.int64(100),
 'amount_usd': np.int64(0),
 'fee': np.int64(90),
 'device_trust_score': np.int64(190),
 'txn_velocity_1h': np.int64(190),
 'txn_velocity_24h': np.int64(0),
 'risk_score_internal': np.int64(0)}

In [42]:
customer_transaction_data = customer_transaction_data.drop(customer_transaction_data[
    (customer_transaction_data['amount_src'] < 0) |
    (customer_transaction_data['fee'] < 0) |
    (customer_transaction_data['device_trust_score'] < 0) |
    (customer_transaction_data['txn_velocity_1h'] < 0)].index)  

In [44]:
# rechecking negative values in numeric columns

negative_counts = {
    'amount_src': (customer_transaction_data['amount_src'] < 0).sum(),
    'amount_usd': (customer_transaction_data['amount_usd'] < 0).sum(),
    'fee': (customer_transaction_data['fee'] < 0).sum(),
    'device_trust_score': (customer_transaction_data['device_trust_score'] < 0).sum(),
    'txn_velocity_1h': (customer_transaction_data['txn_velocity_1h'] < 0).sum(),
    'txn_velocity_24h': (customer_transaction_data['txn_velocity_24h'] < 0).sum(),
    'risk_score_internal': (customer_transaction_data['risk_score_internal'] < 0).sum(), 
    }
negative_counts  

{'amount_src': np.int64(0),
 'amount_usd': np.int64(0),
 'fee': np.int64(0),
 'device_trust_score': np.int64(0),
 'txn_velocity_1h': np.int64(0),
 'txn_velocity_24h': np.int64(0),
 'risk_score_internal': np.int64(0)}

In [45]:
# checking for futuristic timestamp

customer_transaction_data[customer_transaction_data['timestamp'] > pd.Timestamp.utcnow()] 

,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,...,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud


In [46]:
# checking for location mismatch

customer_transaction_data['location_mismatch'].value_counts().reset_index() 

,location_mismatch,count
0,False,9047
1,True,1793


In [47]:
# reviewing the channel variable

customer_transaction_data['channel'].unique() 

array(['ATM', 'web', 'mobile', 'WEB', ' web  ', 'MOBILE', 'unknown',
       'mobille', ' mobile  ', 'weeb', 'ATm', ' ATM  '], dtype=object)

In [48]:
# formatting the channel variable

customer_transaction_data['channel'] = customer_transaction_data['channel'].str.lower().str.strip()
customer_transaction_data['channel'].unique() 

array(['atm', 'web', 'mobile', 'unknown', 'mobille', 'weeb'], dtype=object)

In [51]:
customer_transaction_data['channel'] = customer_transaction_data['channel'].replace({
    'weeb': 'web',
    'mobille': 'mobile'
     
})  

In [53]:
customer_transaction_data['channel'] = customer_transaction_data['channel'].replace({'unknown':np.nan}) 
customer_transaction_data['channel'].unique() 

array(['atm', 'web', 'mobile', nan], dtype=object)

In [54]:
# reviewing the source currency variable

customer_transaction_data['source_currency'].unique() 

array(['USD', 'CAD', 'GBP'], dtype=object)

In [55]:
# reviewing the destination currency variable

customer_transaction_data['dest_currency'].unique()  

array(['CAD', 'MXN', 'CNY', 'EUR', 'INR', 'GBP', 'PHP', 'NGN', 'USD'],
      dtype=object)

In [57]:
# reviewing the kyc_tier variable

customer_transaction_data['kyc_tier'].unique()   

array(['standard', 'enhanced', 'low', ' standard  ', 'standrd',
       ' enhanced  ', 'STANDARD', 'unknown', 'enhancd', ' low  ',
       'ENHANCED', 'LOW'], dtype=object)

In [58]:
# formatting the kyc_tier variable

customer_transaction_data['kyc_tier'] = customer_transaction_data['kyc_tier'].str.lower().str.strip()
customer_transaction_data['kyc_tier'].unique()  

array(['standard', 'enhanced', 'low', 'standrd', 'unknown', 'enhancd'],
      dtype=object)

In [ ]:
customer_transaction_data['kyc_tier'] = customer_transaction_data['kyc_tier'].replace({
    'standrd': 'standard',
    'enhancd': 'enhanced',        
})  

In [60]:
customer_transaction_data['kyc_tier'] = customer_transaction_data['kyc_tier'].replace({'unknown':np.nan}) 
customer_transaction_data['kyc_tier'].unique()  

array(['standard', 'enhanced', 'low', nan], dtype=object)

In [61]:
# reviewing home_country variable

customer_transaction_data['home_country'].unique()   

array(['US', 'CA', 'UK', ' UK  ', ' US  ', 'unknown', ' CA  '],
      dtype=object)

In [62]:
# formatting the home_country variable

customer_transaction_data['home_country'] = customer_transaction_data['home_country'].str.lower().str.strip()
customer_transaction_data['home_country'].unique()   

array(['us', 'ca', 'uk', 'unknown'], dtype=object)

In [63]:
customer_transaction_data['home_country'] = customer_transaction_data['home_country'].replace({'unknown':np.nan}) 
customer_transaction_data['home_country'].unique()   

array(['us', 'ca', 'uk', nan], dtype=object)

In [64]:
# reviewing ip_country variable

customer_transaction_data['ip_country'].unique()    

array(['US', 'CA', 'UK', ' US  ', 'unknown', ' CA  ', ' UK  '],
      dtype=object)

In [65]:
# formatting the ip_country variable

customer_transaction_data['ip_country'] = customer_transaction_data['ip_country'].str.lower().str.strip()
customer_transaction_data['ip_country'].unique()  

array(['us', 'ca', 'uk', 'unknown'], dtype=object)

In [66]:
customer_transaction_data['ip_country'] = customer_transaction_data['ip_country'].replace({'unknown':np.nan}) 
customer_transaction_data['ip_country'].unique()  

array(['us', 'ca', 'uk', nan], dtype=object)

In [76]:
customer_transaction_data.isnull().sum()    

transaction_id                0
customer_id                   0
timestamp                     0
home_country                 32
source_currency               0
dest_currency                 0
channel                      36
amount_src                    0
amount_usd                    0
fee                           0
exchange_rate_src_to_dest     0
device_id                     0
new_device                    0
ip_address                    0
ip_country                   31
location_mismatch             0
ip_risk_score                 0
kyc_tier                     29
account_age_days              0
device_trust_score            0
chargeback_history_count      0
risk_score_internal           0
txn_velocity_1h               0
txn_velocity_24h              0
corridor_risk                 0
is_fraud                      0
dtype: int64

In [77]:
customer_transaction_data.dropna(inplace=True)
customer_transaction_data.info(0 )

<class 'pandas.core.frame.DataFrame'>
Index: 10731 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype              
---  ------                     --------------  -----              
 0   transaction_id             10731 non-null  object             
 1   customer_id                10731 non-null  object             
 2   timestamp                  10731 non-null  datetime64[ns, UTC]
 3   home_country               10731 non-null  object             
 4   source_currency            10731 non-null  object             
 5   dest_currency              10731 non-null  object             
 6   channel                    10731 non-null  object             
 7   amount_src                 10731 non-null  float64            
 8   amount_usd                 10731 non-null  float64            
 9   fee                        10731 non-null  float64            
 10  exchange_rate_src_to_dest  10731 non-null  float64            
 11  device_

In [78]:
customer_transaction_data.head() 

,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,...,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud
0,fee8542d-8ee6-4b0d-9671-c294dd08ed26,402cccc9-28de-45b3-9af7-cc5302aa1f93,2022-10-03 18:40:59.468549+00:00,us,USD,CAD,atm,278.19,278.19,4.25,...,0.123,standard,263,0.522,0,0.223,0,0,0.0,0
1,bfdb9fc1-27fe-4a85-b043-4d813d679259,67c2c6b3-ef0a-4777-a3f1-c84a851bb6ad,2022-10-03 20:39:38.468549+00:00,ca,CAD,MXN,web,208.51,154.29,4.24,...,0.569,standard,947,0.475,0,0.268,0,1,0.0,0
2,fc855034-3ea5-4993-9afa-b511d93fe5e8,6d0d9b27-fa26-45f8-93b1-2df29d182d9c,2022-10-03 23:02:43.468549+00:00,us,USD,CNY,mobile,160.33,160.33,2.70,...,0.437,enhanced,367,0.939,0,0.176,0,0,0.0,0
3,2cf8c08e-42ec-444d-a755-34b9a2a0a4ca,7bd5200c-5d19-44f0-9afe-8b339a05366b,2022-10-04 01:08:53.468549+00:00,us,USD,EUR,mobile,59.41,59.41,2.22,...,0.594,standard,147,0.551,0,0.391,0,0,0.0,0
4,d907a74d-b426-438d-97eb-dbe911aca91c,70a93d26-8e3a-4179-900c-a4a7a74d08e5,2022-10-04 09:35:03.468549+00:00,us,USD,INR,mobile,200.96,200.96,3.61,...,0.121,enhanced,257,0.894,0,0.257,0,0,0.0,0


In [ ]:
import os


os.makedirs("../Finlora_Dataset/artifacts")  

In [72]:
customer_transaction_data.to_csv(r'../Finlora_Dataset/artifacts/Cleaned_Data.csv')     